<a href="https://colab.research.google.com/github/Bavesh-08/fake-news-classifier-using-LSTM/blob/main/Fake_News_Classifier_Using_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

pf=pd.read_csv('/content/fake_news.csv', on_bad_lines='skip', engine='python')

pf.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [ ]:
# checking is there is any null values

pf.isnull().sum()

,0
id,0
title,64
author,244
text,6
label,0


In [ ]:
# droping the null values

pf=pf.dropna()

In [ ]:
# spliting the dependent and independent values

x=pf.drop('label',axis=1)

y=pf['label']

In [ ]:
# importing the libraryies needed for NN

import tensorflow as tf
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

In [ ]:
# assigning the voloobary size

voc_size=10000

In [ ]:
messages=x.copy()

messages['title'][1]


'FLYNN: Hillary Clinton, Big Woman on Campus - Breitbart'

In [ ]:
messages.reset_index(inplace=True)

In [ ]:
import nltk
import re
from nltk.corpus import stopwords

In [ ]:
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
### Dataset Preprocessing
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
    print(i)
    review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
    review = review.lower()
    review = review.split()

    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [ ]:
corpus

['hous dem aid even see comey letter jason chaffetz tweet',
 'flynn hillari clinton big woman campu breitbart',
 'truth might get fire',
 'civilian kill singl us airstrik identifi',
 'iranian woman jail fiction unpublish stori woman stone death adulteri',
 'jacki mason hollywood would love trump bomb north korea lack tran bathroom exclus video breitbart',
 'beno hamon win french socialist parti presidenti nomin new york time',
 'back channel plan ukrain russia courtesi trump associ new york time',
 'obama organ action partner soro link indivis disrupt trump agenda',
 'bbc comedi sketch real housew isi caus outrag',
 'russian research discov secret nazi militari base treasur hunter arctic photo',
 'us offici see link trump russia',
 'ye paid govern troll social media blog forum websit',
 'major leagu soccer argentin find home success new york time',
 'well fargo chief abruptli step new york time',
 'anonym donor pay million releas everyon arrest dakota access pipelin',
 'fbi close hilla

In [ ]:
# onehot encoding

onehot_repr=[one_hot(words,voc_size)for words in corpus]
onehot_repr

[[4217, 9263, 7782, 7195, 410, 2187, 8888, 1864, 3865, 7743],
 [2398, 5959, 2908, 6234, 5828, 1860, 4942],
 [2677, 3503, 3838, 5014],
 [6500, 1635, 6655, 4038, 9901, 1103],
 [4717, 5828, 5803, 3544, 4350, 9573, 5828, 8047, 4662, 9451],
 [6341,
  317,
  9452,
  6764,
  6893,
  7709,
  253,
  1059,
  8566,
  7524,
  6836,
  5639,
  5340,
  6199,
  4942],
 [7853, 1180, 927, 5294, 9475, 9739, 5979, 2872, 859, 1366, 6271],
 [8635, 7718, 1441, 7938, 8882, 3255, 7709, 3828, 859, 1366, 6271],
 [3835, 6949, 6212, 7784, 1200, 1031, 9787, 3850, 7709, 4239],
 [6962, 163, 1619, 8721, 9562, 1493, 3979, 1981],
 [7883, 9551, 4717, 8431, 4912, 2116, 8073, 5457, 2427, 3209, 1670],
 [4038, 2573, 410, 1031, 7709, 8882],
 [6056, 2800, 5276, 1716, 8313, 8696, 728, 7130, 8550],
 [519, 6363, 5647, 6019, 7045, 1327, 4679, 859, 1366, 6271],
 [1304, 6424, 1529, 6424, 1586, 859, 1366, 6271],
 [29, 2183, 368, 9772, 6859, 3444, 6557, 4646, 886, 9861],
 [3461, 9546, 5959],
 [4458, 307, 3182, 6529, 7709, 7499, 8862, 

In [ ]:
# embedding part

sent_length=20
embedded_docs=pad_sequences(onehot_repr,padding='pre',maxlen=sent_length)
print(embedded_docs)

[[   0    0    0 ... 1864 3865 7743]
 [   0    0    0 ... 5828 1860 4942]
 [   0    0    0 ... 3503 3838 5014]
 ...
 [   0    0    0 ... 8313 2558 4942]
 [   0    0    0 ...  859 1366 6271]
 [   0    0    0 ... 1508  290 1692]]


In [ ]:
embedded_docs[0]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0, 4217,
       9263, 7782, 7195,  410, 2187, 8888, 1864, 3865, 7743], dtype=int32)

In [ ]:
## Creating model
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
len(embedded_docs),y.shape

(2084, (2084,))

In [ ]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [ ]:
X_final.shape,y_final.shape

((2084, 20), (2084,))

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

In [ ]:
### Finally Training
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=100,batch_size=64)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 93ms/step - accuracy: 0.7099 - loss: 0.6265 - val_accuracy: 0.8140 - val_loss: 0.4837
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.8417 - loss: 0.3683 - val_accuracy: 0.8677 - val_loss: 0.3168
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.9234 - loss: 0.2334 - val_accuracy: 0.8837 - val_loss: 0.2616
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.9484 - loss: 0.1420 - val_accuracy: 0.8852 - val_loss: 0.2692
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.9850 - loss: 0.0643 - val_accuracy: 0.8953 - val_loss: 0.2901
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.9928 - loss: 0.0295 - val_accuracy: 0.8968 - val_loss: 0.2682
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.9936 - loss: 0.0192 - val_accuracy: 0.8997 - val_loss: 0.2594
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.9986 - loss: 0.0081 - val_accuracy: 0.

In [ ]:
from tensorflow.keras.layers import Dropout
## Creating model
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(Dropout(0.3))
model.add(LSTM(100))
model.add(Dropout(0.3))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")

22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step


In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
confusion_matrix(y_test,y_pred)

array([[344,  47],
       [221,  76]])

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.6104651162790697